<a href="https://colab.research.google.com/github/everestso/Atari_2600_Games/blob/main/a2c_Breakout_Replay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install packages
!pip uninstall -y gym
!pip install -U "gymnasium[atari]" "stable-baselines3[extra]"

Found existing installation: gym 0.25.2
Uninstalling gym-0.25.2:
  Successfully uninstalled gym-0.25.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 12.5 MB/s eta 0:00:00


In [4]:
import os
import urllib.request

# -----------------------------
# Game configurations
# -----------------------------
games = {
    "pong": {
        "game_name": "Pong",
        "env_id": "ALE/Pong-v5",
        "model_name": "a2c_pong_model",
    },
    "breakout": {
        "game_name": "Breakout",
        "env_id": "ALE/Breakout-v5",
        "model_name": "a2c_breakout_model",
    },
}

# -----------------------------
# Select game and model
# -----------------------------
selected_game = "breakout"   # change to "pong" or "breakout"
cfg = games[selected_game]

# GitHub raw download URL
github_raw_url = (
    "https://raw.githubusercontent.com/everestso/Atari_2600_Games/main/"
    "Models/Breakout/a2c_breakout_model_6200000.zip"
)

# -----------------------------
# Local paths
# -----------------------------
model_dir = "/content/models"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, model_name)

video_folder = os.path.join("videos", "breakout")
os.makedirs(video_folder, exist_ok=True)

# -----------------------------
# Download model from GitHub
# -----------------------------
if not os.path.exists(model_path):
    print(f"Downloading Breakout model to: {model_path}")
    urllib.request.urlretrieve(github_raw_url, model_path)
    print("Download complete.")
else:
    print("Model already exists locally.")

print(f"{model_path=}")

Model already exists locally.
model_path='/content/models/a2c_breakout_model_6200000.zip'


In [5]:

import glob
import time
import base64
from IPython.display import HTML, display

import gymnasium as gym
import ale_py

gym.register_envs(ale_py)

from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, VecVideoRecorder

seed = 0
n_stack = 4
terminal_on_life_loss = False
video_length = 10000
max_seconds = 30

print("Selected game:", cfg["game_name"])
print("Environment ID:", cfg["env_id"])
print("Model path:", model_path)
print("Video folder:", video_folder)

# -----------------------------
# Check model exists
# -----------------------------
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file not found: {model_path}")

# -----------------------------
# Optional cleanup
# -----------------------------
for f in glob.glob(os.path.join(video_folder, "*.mp4")):
    os.remove(f)

# -----------------------------
# Load model
# -----------------------------
model = A2C.load(model_path)

# -----------------------------
# Build SAME preprocessing pipeline used for training
# -----------------------------
eval_env = make_atari_env(
    cfg["env_id"],
    n_envs=1,
    seed=seed,
    wrapper_kwargs=dict(terminal_on_life_loss=terminal_on_life_loss),
)
eval_env = VecFrameStack(eval_env, n_stack=n_stack)

# -----------------------------
# Record video
# -----------------------------
eval_env = VecVideoRecorder(
    eval_env,
    video_folder=video_folder,
    record_video_trigger=lambda step: step == 0,
    video_length=video_length,
    name_prefix=cfg["model_name"]
)

obs = eval_env.reset()

# Additional wall-clock safety cap
start_time = time.time()
step_count = 0
done = [False]

while (
    step_count < video_length
    and (time.time() - start_time) < max_seconds
    and not done[0]
):
    action, _states = model.predict(obs, deterministic=True)
    obs, rewards, done, infos = eval_env.step(action)
    step_count += 1

eval_env.close()

elapsed = time.time() - start_time
print(f"Stopped after {step_count} steps and {elapsed:.1f} seconds.")

# -----------------------------
# Display newest video
# -----------------------------
video_files = glob.glob(os.path.join(video_folder, "*.mp4"))

if video_files:
    video_path = sorted(video_files)[-1]
    print("Video created:", video_path)

    with open(video_path, "rb") as f:
        mp4 = f.read()

    data_url = "data:video/mp4;base64," + base64.b64encode(mp4).decode()

    display(HTML(f"""
    <div style="width: 100%; overflow-x: auto;">
        <video controls playsinline style="width: 100%; max-width: 360px; height: auto;">
            <source src="{data_url}" type="video/mp4">
        </video>
    </div>
    """))
else:
    print("No video file was created.")

Selected game: Breakout
Environment ID: ALE/Breakout-v5
Model path: /content/models/a2c_breakout_model_6200000.zip
Video folder: videos/breakout


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Moviepy - Building video /content/videos/breakout/a2c_breakout_model-step-0-to-step-10000.mp4.
Moviepy - Writing video /content/videos/breakout/a2c_breakout_model-step-0-to-step-10000.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/breakout/a2c_breakout_model-step-0-to-step-10000.mp4
Stopped after 174 steps and 2.0 seconds.
Video created: videos/breakout/a2c_breakout_model-step-0-to-step-10000.mp4
